In [1]:
import pandas as pd

In [3]:
# Load CSV
df = pd.read_csv("C:/Chen Liwei/techlent_2024/learnpy/LLM/Mychatbot-example/fullstack_flask/src/data/Mental_Health_FAQ.csv")

# Convert to JSON format for easier processing
data = df.to_dict(orient="records")

# Save JSON file
with open("mental_health_resources.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)

print("Data successfully saved in JSON format!")

Data successfully saved in JSON format!


In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Question_ID  98 non-null     int64 
 1   Questions    98 non-null     object
 2   Answers      98 non-null     object
dtypes: int64(1), object(2)
memory usage: 2.4+ KB


In [7]:
df.head()


,Question_ID,Questions,Answers
0,1590140,What does it mean to have a mental illness?,Mental illnesses are health conditions that di...
1,2110618,Who does mental illness affect?,It is estimated that mental illness affects 1 ...
2,6361820,What causes mental illness?,It is estimated that mental illness affects 1 ...
3,9434130,What are some of the warning signs of mental i...,Symptoms of mental health disorders vary depen...
4,7657263,Can people with mental illness recover?,"When healing from mental illness, early identi..."


In [9]:
pip install -U langchain langchain-openai langchain-community pinecone-client pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
import pandas as pd
from tqdm import tqdm
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import Pinecone as PineconeVectorStore

# Load your API keys
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  
PINECONE_INDEX_NAME = "mental-health-chatbot"

# Initialize Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

# Check if index exists, create if not
if PINECONE_INDEX_NAME in pc.list_indexes().names():
    print(f"Index '{PINECONE_INDEX_NAME}' already exists. Deleting it...")
    pc.delete_index(PINECONE_INDEX_NAME)

# Create the index
print(f"Creating index '{PINECONE_INDEX_NAME}'...")
pc.create_index(name=PINECONE_INDEX_NAME, dimension=1536, metric="cosine", spec=ServerlessSpec(cloud="aws", region="us-east-1"))

# Connect to the Pinecone index
index = pc.Index(PINECONE_INDEX_NAME)

# Load the dataset
df = pd.read_csv("C:/Chen Liwei/techlent_2024/learnpy/LLM/Mychatbot-example/fullstack_flask/src/data/Mental_Health_FAQ.csv")

# Initialize OpenAI embeddings model
embeddings = OpenAIEmbeddings(model='text-embedding-ada-002')

# Create Pinecone VectorStore
vectorstore = PineconeVectorStore(index=index, embedding=embeddings)

# Upload Data to Pinecone
for _, row in tqdm(df.iterrows(), total=df.shape[0]):
    question = row["Questions"]
    answer = row["Answers"]
    vector = embeddings.embed_query(question)  # Convert question into vector

    # Store in Pinecone with metadata
    index.upsert([
        (str(row["Question_ID"]), vector, {"question": question, "answer": answer})
    ])

print("✅ Data successfully uploaded to Pinecone!")

Index 'mental-health-chatbot' already exists. Deleting it...
Creating index 'mental-health-chatbot'...


100%|████████████████████████████████████████████████████████████████████████████████| 103/103 [01:57<00:00,  1.14s/it]

✅ Data successfully uploaded to Pinecone!
